In [2]:
import os
import scanpy as sc
import pandas as pd
EMBEDDING_DIR =  '/media/lleger/LaCie/mit/disease_vector/'
h5ad_files = [os.path.join(EMBEDDING_DIR, f) for f in os.listdir(EMBEDDING_DIR) if f.endswith(".h5ad")]

adata_list = [sc.read_h5ad(f) for f in sorted(h5ad_files)]
adata = adata_list[0].concatenate(adata_list[1:], index_unique=None)

adata.obs_names = [str(i) for i in range(adata.n_obs)]

sampled_indices = (
    adata.obs.groupby("disease", group_keys=False)
    .apply(lambda x: x.sample(min(50, len(x)), random_state=0))
    .index
)
adata = adata[sampled_indices, :]

adata.write_h5ad("../../../data/test_cxg_disease_balanced.h5ad")

/tmp/ipykernel_3258098/668084441.py:8: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adata_list[0].concatenate(adata_list[1:], index_unique=None)


## Glioblastoma Drug Screen

In [1]:
EMBEDDING_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/glioblastoma_study/'
RAW_DIR = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW"
import pandas as pd
metadata_matrix = pd.read_csv("GSE148842-GPL18573_series_matrix.txt", sep='\t', comment='!', header=None, index_col=0).T
metadata_matrix.columns = ['age', 'gender', 'tissue', 'disease', 'drug', 'protocol', 'filename']
metadata_matrix


,age,gender,tissue,disease,drug,protocol,filename
1,age: 52,gender: F,location: splenial extension into left parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: vehicle (DMSO),Tumor specimens were collected immediately aft...,GSM4483741
2,age: 52,gender: F,location: splenial extension into left parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 2.5 uM etoposide,Tumor specimens were collected immediately aft...,GSM4483742
3,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: vehicle (DMSO),Tumor specimens were collected immediately aft...,GSM4483743
4,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: vehicle (DMSO),Tumor specimens were collected immediately aft...,GSM4483744
5,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 2.5 uM etoposide,Tumor specimens were collected immediately aft...,GSM4483745
6,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 0.2 uM panobinostat,Tumor specimens were collected immediately aft...,GSM4483746
7,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 50 nM RO4929097,Tumor specimens were collected immediately aft...,GSM4483747
8,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 50 uM Tazemetostat,Tumor specimens were collected immediately aft...,GSM4483748
9,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 1.8 nM Ispenisib,Tumor specimens were collected immediately aft...,GSM4483749
10,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 40 nM Ana-12,Tumor specimens were collected immediately aft...,GSM4483750


In [2]:
import sys
sys.path.append('../../../')
from polygene.model.model import load_trained_model
from polygene.data_utils.tokenization import normalise_str
EMBEDDING_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/glioblastoma_study/'
data_path = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW/"

m, tok = load_trained_model("../../../runs/gesam_polygene_run_4/")
import json
import scanpy as sc
import os
import scipy.sparse as sp

from tqdm import tqdm
known_genes = set(list(json.load(open("../../data_utils/vocab/gene_ranking_map.json")).keys())) # checked overlap aftering dropping '.' in ENSEMBL ID

pbar = tqdm(os.listdir(data_path)[15:])
for file in pbar:
    drug = metadata_matrix[metadata_matrix['filename'] == file.split('_')[0]]['drug'].tolist()
    if not drug:
        print('ok')
        continue
    pbar.set_description(f'Formatting study to anndata: {drug}')
    df = pd.read_csv(data_path + file, sep="\t", index_col=0)
    df = df.drop(columns=df.columns[0])
    df.index = pd.Series(df.index.tolist()).apply(lambda x: x.split('.')[0]).tolist()
    
    adata = sc.AnnData(df.T)
    adata.obs['drug'] = drug * len(adata)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.X = sp.csr_matrix(adata.X)
    adata.write_h5ad(EMBEDDING_DIR + f"{file.split('.')[0]}.h5ad")

/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Formatting study to anndata: ['treatment: vehicle (DMSO)']:  11%|█         | 3/28 [00:11<01:39,  3.97s/it]     

ok
ok
ok
ok


Formatting study to anndata: ['treatment: 0.2 uM panobinostat']: 100%|██████████| 28/28 [02:22<00:00,  5.09s/it]

ok
ok
ok
ok
ok
ok
ok
ok
ok
ok


In [3]:
# Load glioblastoma from CxG
import sys
sys.path.append('../../../')
import numpy as np
import sys
import pandas as pd
sys.path.append('../../../')
from polygene.model.model import load_trained_model
from polygene.data_utils.tokenization import normalise_str
EMBEDDING_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/glioblastoma_study/'
data_path = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW/"
m, tok = load_trained_model("../../../runs/gesam_polygene_run_4/")

cxg_embeddings = pd.read_pickle(EMBEDDING_DIR + "../glioblastoma_embeddings.pkl")

normal_embeddings = cxg_embeddings[0][cxg_embeddings[2][:, tok.phenotypic_types.index('disease')] == "[normal]"]
disease_embeddings = cxg_embeddings[0][cxg_embeddings[2][:, tok.phenotypic_types.index('disease')] != "[normal]"]
a, b = np.mean(normal_embeddings, axis=0), np.mean(disease_embeddings, axis=0)

/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
pd.Series(cxg_embeddings[2][:, tok.phenotypic_types.index('disease')]).value_counts(), disease_embeddings.shape

([glioblastoma]    9432
 [normal]          5117
 dtype: int64,
 (9432, 256))

In [ ]:
from polygene.eval.metrics import prepare_cell
import torch
from tqdm import tqdm
import os
import scanpy as sc
embeddings, labels, predictions, drug = ([] for _ in range(4))
for path in os.listdir(EMBEDDING_DIR):
    if path.endswith('pkl'): continue
    ad = sc.read_h5ad(EMBEDDING_DIR + path)#[:1000]
    for cell in tqdm(ad):
        cell_dict = prepare_cell(cell, tok)
        cell_dict['input_ids'][np.arange(1, 1 + len(tok.phenotypic_types))] = 2
        with torch.no_grad():
            output = m(**{key: val.to(m.device).unsqueeze(0) for key, val in cell_dict.items() if key != 'str_labels'})
        encoder_output = output.hidden_states
        embeddings.append(encoder_output[0, 1 + tok.phenotypic_types.index('disease')].detach().cpu().numpy())
        labels.append(cell_dict['str_labels'][1:1 + len(tok.phenotypic_types)])
        predictions.append([tok.flattened_tokens[output.logits.argmax(dim=-1).squeeze()[1 + idx]] 
                    for idx in range(len(tok.phenotypic_types))])
    drug.extend( metadata_matrix[metadata_matrix['filename'] == path.split('_')[0]]['drug'].tolist() * len(ad))
    
df_g = pd.DataFrame({"embeddings": embeddings, "labels": labels, "predictions": predictions, "drug": drug})
df_g.to_pickle(EMBEDDING_DIR + "embeddings.pkl")

In [ ]:
# Analysis of df_g
import numpy as np
df_g = pd.read_pickle(EMBEDDING_DIR + "embeddings.pkl")
df_g['disease'] = np.array(df_g['predictions'].tolist())[:, tok.phenotypic_types.index('disease')]



v = b - a 
v /= np.dot(v,v)


s_a = np.array(df_g[(df_g['disease'] == "[normal]") & (df_g['drug'] == "treatment: none")]['embeddings'].tolist()).mean(axis=0)

df_g = df_g[df_g['disease'] == '[glioblastoma]']

study_ranking = drug_effectiveness_rank = {
    "treatment: 0.2 uM panobinostat": 1,
    "treatment: 2.5 uM etoposide": 2,
    "treatment: 1.8 nM Ispenisib": 3,
    "treatment: 50 uM Tazemetostat": 4,
    "treatment: 50 nM RO4929097": 5,
    "treatment: 40 nM Ana-12": 6,
    "treatment: vehicle (DMSO)": 7,
    "treatment: none": 8
}
display(df_g.value_counts(['disease', 'drug']).reset_index())
centroid_study = df_g.groupby('drug').apply(lambda g: pd.Series({'centroid': np.array(g['embeddings']).mean(axis=0)})).reset_index()
centroid_study['Disease Vector Position'] = centroid_study['centroid'].apply(lambda c: (c - a) @ v)
centroid_study['shifted'] = centroid_study['centroid'].apply(lambda c: c - centroid_study[centroid_study['drug'] == 'treatment: none']['centroid'].values[0])
centroid_study['Cosine Similarity of centroid vectors'] = centroid_study['centroid'].apply(lambda s: (np.dot(s-s_a, b-a))/(np.linalg.norm(b-a) * np.linalg.norm(s-s_a)))
centroid_study['Study Ranking'] = centroid_study['drug'].apply(lambda x: study_ranking.get(x,x))
centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Disease Vector Position')
centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Cosine Similarity of centroid vectors')

,disease,drug,0
0,[glioblastoma],treatment: vehicle (DMSO),37245
1,[glioblastoma],treatment: 2.5 uM etoposide,12655
2,[glioblastoma],treatment: 0.2 uM panobinostat,8049
3,[glioblastoma],treatment: none,5299
4,[glioblastoma],treatment: 40 nM Ana-12,2732
5,[glioblastoma],treatment: 50 nM RO4929097,2341
6,[glioblastoma],treatment: 50 uM Tazemetostat,1926
7,[glioblastoma],treatment: 1.8 nM Ispenisib,1174


,drug,Disease Vector Position,Cosine Similarity of centroid vectors,Study Ranking
5,treatment: 50 uM Tazemetostat,0.685100,0.391291,4
0,treatment: 0.2 uM panobinostat,0.695570,0.413382,1
4,treatment: 50 nM RO4929097,0.709641,0.415369,5
1,treatment: 1.8 nM Ispenisib,0.697789,0.419223,3
3,treatment: 40 nM Ana-12,0.720375,0.453856,6
2,treatment: 2.5 uM etoposide,0.711461,0.462677,2
7,treatment: vehicle (DMSO),0.708690,0.478995,7
6,treatment: none,0.690890,0.600090,8


In [ ]:
centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Disease Vector Position')
# features related to the change of state. 

,drug,Disease Vector Position,Cosine Similarity of centroid vectors,Study Ranking
5,treatment: 50 uM Tazemetostat,0.685100,0.391291,4
6,treatment: none,0.690890,0.600090,8
0,treatment: 0.2 uM panobinostat,0.695570,0.413382,1
1,treatment: 1.8 nM Ispenisib,0.697789,0.419223,3
7,treatment: vehicle (DMSO),0.708690,0.478995,7
4,treatment: 50 nM RO4929097,0.709641,0.415369,5
2,treatment: 2.5 uM etoposide,0.711461,0.462677,2
3,treatment: 40 nM Ana-12,0.720375,0.453856,6


In [8]:
df_g = pd.DataFrame({"embeddings": embeddings, "labels": labels, "predictions": predictions, "drug": drug})
df_g.to_pickle(EMBEDDING_DIR + "embeddings.pkl")

In [ ]:

df_g['disease'] = np.array(df_g['predictions'].tolist())[:, tok.phenotypic_types.index('disease')]

NameError: name 'df' is not defined